# DistilBERT extension: data and tokenization

Prepare the existing GoEmotions sentiment split for DistilBERT. This uses raw `text`, the same 680/120 train/validation split as the neural experiments, and leaves the held-out test split untouched until evaluation. Run from the repository root in Colab or another environment with the project files available.

In [ ]:
%pip install -q "transformers>=4.45,<6.0" "torch>=2.4,<3.0" pandas scikit-learn

from pathlib import Path
import math
import sys

candidates = [Path.cwd(), *Path.cwd().parents]
ROOT_DIR = next((path for path in candidates if (path / 'src' / 'preprocessing.py').exists()), None)
if ROOT_DIR is None:
    raise FileNotFoundError('Start from the repository root or place the repository in the current directory.')
sys.path.insert(0, str(ROOT_DIR))

from src.preprocessing import SEED, load_splits, train_val_split
from src.models import build_transformer
from src.evaluate import compute_metrics, metrics_row, save_metrics
from transformers import AutoTokenizer
import numpy as np

MODEL_NAME = 'distilbert-base-uncased'
MAX_LENGTH_CAP = 64
SEED

## Load the shared split

`load_splits` defaults to cleaned text, so pass `text_col='text'` to give DistilBERT the original comments. Only the training portion determines the validation split and token-length limit.

In [ ]:
X_train_all, y_train_all, X_test, y_test = load_splits(text_col='text')
X_train, X_val, y_train, y_val = train_val_split(X_train_all, y_train_all, seed=SEED)
print(f'train={len(X_train)}, validation={len(X_val)}, test={len(X_test)}')
print(f'train labels={np.bincount(y_train)}, validation labels={np.bincount(y_val)}')
assert (len(X_train), len(X_val), len(X_test)) == (680, 120, 200)
assert set(np.unique(y_train_all)) == {0, 1}

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
train_lengths = [len(tokenizer.encode(text, add_special_tokens=True)) for text in X_train]
p99_length = int(math.ceil(np.quantile(train_lengths, 0.99)))
MAX_LENGTH = MAX_LENGTH_CAP if p99_length <= MAX_LENGTH_CAP else p99_length
train_truncation_rate = float(np.mean(np.asarray(train_lengths) > MAX_LENGTH))
print(f'train subword lengths: p99={p99_length}, max={max(train_lengths)}')
print(f'max_length={MAX_LENGTH}; training truncation rate={train_truncation_rate:.1%}')
if p99_length > MAX_LENGTH_CAP:
    print('The training p99 exceeds 64 tokens, so max_length was increased to cover the training p99.')

## Tokenize without fitting on test data

The pretrained tokenizer is fixed, so no vocabulary is learned from this dataset. The maximum sequence length is selected from training comments only. Test comments are tokenized with that fixed setting for later evaluation; they are not used to choose it.

In [ ]:
def tokenize_texts(texts):
    return tokenizer(
        list(texts),
        truncation=True,
        padding='max_length',
        max_length=MAX_LENGTH,
        return_attention_mask=True,
    )

tokenized_train = tokenize_texts(X_train)
tokenized_val = tokenize_texts(X_val)
tokenized_test = tokenize_texts(X_test)
for name, batch in [('train', tokenized_train), ('validation', tokenized_val), ('test', tokenized_test)]:
    ids = np.asarray(batch['input_ids'])
    masks = np.asarray(batch['attention_mask'])
    assert ids.shape == masks.shape == (len(batch['input_ids']), MAX_LENGTH)
    print(f'{name}: input_ids={ids.shape}, attention_mask={masks.shape}')

## Fine-tune DistilBERT (J2)

The Trainer evaluates once per epoch and restores the checkpoint with the highest validation macro F1. The held-out test split is used only after training is complete. Checkpoints are written to a temporary directory and are not added to the repository.

In [ ]:
from tempfile import TemporaryDirectory
import torch
from torch.utils.data import Dataset
from transformers import Trainer, TrainingArguments, set_seed

class SentimentDataset(Dataset):
    def __init__(self, tokenized, labels):
        self.input_ids = tokenized['input_ids']
        self.attention_mask = tokenized['attention_mask']
        self.labels = np.asarray(labels, dtype=np.int64)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        return {
            'input_ids': torch.tensor(self.input_ids[index], dtype=torch.long),
            'attention_mask': torch.tensor(self.attention_mask[index], dtype=torch.long),
            'labels': torch.tensor(self.labels[index], dtype=torch.long),
        }

train_dataset = SentimentDataset(tokenized_train, y_train)
val_dataset = SentimentDataset(tokenized_val, y_val)
test_dataset = SentimentDataset(tokenized_test, y_test)

def trainer_metrics(eval_prediction):
    predictions = np.asarray(eval_prediction.predictions)
    labels = np.asarray(eval_prediction.label_ids)
    return compute_metrics(labels, predictions.argmax(axis=-1))

set_seed(SEED)
model = build_transformer(MODEL_NAME)
training_args_config = dict(
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    greater_is_better=True,
    save_total_limit=1,
    logging_strategy='epoch',
    report_to='none',
    seed=SEED,
    data_seed=SEED,
    fp16=torch.cuda.is_available(),
)

with TemporaryDirectory(prefix='distilbert-j2-') as checkpoint_dir:
    training_args = TrainingArguments(output_dir=checkpoint_dir, **training_args_config)
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=trainer_metrics,
    )
    train_result = trainer.train()
    train_predictions = trainer.predict(train_dataset, metric_key_prefix='train')
    test_predictions = trainer.predict(test_dataset, metric_key_prefix='test')
    best_metric = float(trainer.state.best_metric)
    best_epoch = next(
        (entry['epoch'] for entry in trainer.state.log_history
         if entry.get('eval_f1_macro') == best_metric),
        None,
    )

train_time_s = float(train_result.metrics['train_runtime'])
n_params = sum(parameter.numel() for parameter in trainer.model.parameters())
rows = [
    metrics_row('distilbert', 'train', SEED, y_train, train_predictions.predictions.argmax(axis=-1), train_time_s, n_params),
    metrics_row('distilbert', 'test', SEED, y_test, test_predictions.predictions.argmax(axis=-1), train_time_s, n_params),
]
metrics_path = ROOT_DIR / 'outputs' / 'metrics' / 'distilbert.csv'
distilbert_metrics = save_metrics(rows, metrics_path)
print(f'Best validation macro F1: {best_metric:.4f} at epoch {best_epoch}')
print(f'Training time: {train_time_s:.1f}s; parameters: {n_params:,}')
display(distilbert_metrics)